In [29]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing import image
from PIL import ImageFile
from sklearn.model_selection import GridSearchCV
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

print("All modules imported successfully.")


All modules imported successfully.


In [30]:
# ==============================
#  Load dataset
# ==============================
data_dir = "./../data sheets/training_set"

In [32]:
# Allow truncated images (prevents load errors)
ImageFile.LOAD_TRUNCATED_IMAGES = True

In [ ]:
batch_size = 32
img_size = (224, 224)

# Data generator with VGG16 preprocessing
datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=25,
    width_shift_range=0.25,
    height_shift_range=0.25,
    shear_range=0.2,
    zoom_range=0.3,
    brightness_range=[0.8, 1.2],
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=0.2
)

train_gen = datagen.flow_from_directory(
    data_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode="sparse",
    subset='training',
    shuffle=True
)

val_gen = datagen.flow_from_directory(
    data_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode="sparse",
    subset='validation',
    shuffle=False
)





Found 4000 images belonging to 5 classes.
Found 1000 images belonging to 5 classes.


In [34]:
# =======================================================
# STEP 2 — Fine-tune VGG16
# =======================================================
base_model = VGG16(weights="imagenet", include_top=False, input_shape=(224, 224, 3))

# Freeze early layers, unfreeze last 4 convolutional blocks
for layer in base_model.layers[:-4]:
    layer.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.4)(x)
predictions = Dense(5, activation='softmax')(x)  # 5 classes

ft_model = Model(inputs=base_model.input, outputs=predictions)

ft_model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

print("Fine-tuning VGG16...")
ft_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10,
    callbacks=[early_stop],
    verbose=1
)


Fine-tuning VGG16...


c:\Users\MODERN\anaconda3\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 1231s 10s/step - accuracy: 0.2024 - loss: 1.8803 - val_accuracy: 0.2320 - val_loss: 1.6082
Epoch 2/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 537s 4s/step - accuracy: 0.2290 - loss: 1.6094 - val_accuracy: 0.2210 - val_loss: 1.6051
Epoch 3/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 1937s 16s/step - accuracy: 0.2228 - loss: 1.6065 - val_accuracy: 0.2160 - val_loss: 1.6078
Epoch 4/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 4089s 33s/step - accuracy: 0.2140 - loss: 1.6094 - val_accuracy: 0.2310 - val_loss: 1.6057
Epoch 5/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 2276s 18s/step - accuracy: 0.2318 - loss: 1.6039 - val_accuracy: 0.2210 - val_loss: 1.6032
Epoch 6/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 3009s 24s/step - accuracy: 0.2340 - loss: 1.6019 - val_accuracy: 0.2350 - val_loss: 1.6056
Epoch 7/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 833s 7s/step - accuracy: 0.2514 - loss: 1.6013 - val_accuracy: 0.2460 - val_loss: 1.5896
Epoch 8/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 845s 7s/step - accuracy: 0.2728 - loss: 1.5808 

In [ ]:

# =======================================================
# STEP 3 — Feature Extraction using the fine-tuned model
# =======================================================
feature_extractor = Model(inputs=ft_model.input, outputs=ft_model.get_layer("block5_pool").output)

def extract_features(generator, model):
    features = []
    labels = []
    for i in range(len(generator)):
        try:
            x_batch, y_batch = generator[i]
            feat_batch = model.predict(x_batch, verbose=0)
            feat_batch = feat_batch.reshape(feat_batch.shape[0], -1)
            features.append(feat_batch)
            labels.append(y_batch)
            if (i + 1) * batch_size >= generator.n:
                break
        except Exception as e:
            print(f" Skipping batch {i} due to error: {e}")
            continue
    X = np.vstack(features)
    y = np.hstack(labels)
    return X, y

print(" Extracting features from fine-tuned model...")
X_train, y_train = extract_features(train_gen, feature_extractor)
X_test, y_test = extract_features(val_gen, feature_extractor)

print("Feature extraction complete.")
print("Train features:", X_train.shape, " Test features:", X_test.shape)


 Extracting features from fine-tuned model...
Feature extraction complete.
Train features: (4000, 25088)  Test features: (1000, 25088)


In [36]:
# =======================================================
# STEP 4 — Normalize + PCA
# =======================================================
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

pca = PCA(n_components=256, random_state=42)
X_train = pca.fit_transform(X_train)
X_test = pca.transform(X_test)

In [ ]:

# =======================================================
# STEP 5 — Train SVM Classifier
# =======================================================
param_grid = {
    'C': [1, 5, 10, 50, 100],
    'gamma': ['scale', 0.001, 0.0005],
    'kernel': ['rbf']
}

print(" Performing Grid Search for best SVM parameters...")
grid = GridSearchCV(SVC(random_state=42), param_grid, cv=3, n_jobs=-1, verbose=2)
grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)
clf = grid.best_estimator_

# Train final model
print(" Training final SVM with best parameters...")
clf.fit(X_train, y_train)


 Performing Grid Search for best SVM parameters...
Fitting 3 folds for each of 15 candidates, totalling 45 fits
Best Parameters: {'C': 5, 'gamma': 'scale', 'kernel': 'rbf'}
 Training final SVM with best parameters...


SVC(C=5, random_state=42)

In [38]:

# ==============================
# Step 4: Evaluate model
# ==============================
y_pred = clf.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print("\n Accuracy:", round(accuracy * 100, 2), "%")

# Detailed evaluation
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


 Accuracy: 27.8 %

Classification Report:
              precision    recall  f1-score   support

         0.0       0.31      0.21      0.25       200
         1.0       0.31      0.28      0.30       200
         2.0       0.17      0.07      0.09       200
         3.0       0.40      0.26      0.31       200
         4.0       0.24      0.57      0.34       200

    accuracy                           0.28      1000
   macro avg       0.29      0.28      0.26      1000
weighted avg       0.29      0.28      0.26      1000


Confusion Matrix:
[[ 42  35  12  18  93]
 [ 19  56  13   8 104]
 [ 33  37  13  30  87]
 [ 23  24  19  52  82]
 [ 17  26  19  23 115]]


In [28]:
# ==============================
# Step 5: Predict face shape names
# ==============================
class_labels = {v: k for k, v in generator.class_indices.items()}

predicted_shapes = [class_labels[int(p)] for p in y_pred]
print("\n Example Predictions (first 10):")
print(predicted_shapes[:10])


 Example Predictions (first 10):
['Oblong', 'Oval', 'Oval', 'Oval', 'Oval', 'Heart', 'Square', 'Oblong', 'Heart', 'Oval']
